# Módulo de Validación Estadística de Números Pseudoaleatorios
**Simulación por Computador — Universidad Pedagógica y Tecnológica de Colombia**

Este módulo implementa las seis pruebas estadísticas de validación requeridas por el taller:
Prueba de Medias, Prueba de Varianza, Prueba Chi-Cuadrado, Prueba Kolmogorov-Smirnov,
Prueba de Póker y Prueba de Rachas. Todas las pruebas están diseñadas para recibir
secuencias generadas por  generadores pseudoaleatorios

In [ ]:
import math
import matplotlib.pyplot as plt
from scipy.stats import chi2, norm

## 1. Prueba de Medias
Verifica que la media de la secuencia pseudoaleatoria sea estadísticamente
cercana a 0.5, valor esperado de una distribución uniforme U(0,1).

In [ ]:
def prueba_medias(secuencia, nivel_confianza=0.95):
    """
    Parámetros:
        secuencia: lista de números pseudoaleatorios entre 0 y 1
        nivel_confianza: nivel de confianza para la prueba (default 95%)
    Retorna:
        resultado: diccionario con todos los valores calculados
    """
    # Número de elementos en la secuencia
    N = len(secuencia)

    # Calcular la media muestral
    media_muestral = sum(secuencia) / N

    # Valor Z calculado dinámicamente según nivel de confianza
    z = norm.ppf(1 - (1 - nivel_confianza) / 2)

    # Media teórica para U(0,1)
    media_teorica = 0.5

    # Intervalo de confianza
    margen = z * (1 / math.sqrt(12 * N))
    limite_inferior = media_teorica - margen
    limite_superior = media_teorica + margen

    # Verificar si la secuencia pasa la prueba
    aprueba = limite_inferior <= media_muestral <= limite_superior

    return {
        "N": N,
        "media_muestral": round(media_muestral, 6),
        "media_teorica": media_teorica,
        "valor_z": round(z, 6),
        "margen": round(margen, 6),
        "limite_inferior": round(limite_inferior, 6),
        "limite_superior": round(limite_superior, 6),
        "aprueba": aprueba
    }

In [ ]:
def grafico_prueba_medias(resultado):
    """
    Gráfico para una sola secuencia: muestra la media muestral
    frente al intervalo de confianza y la media teórica.

    Parámetros:
        resultado: diccionario retornado por prueba_medias()
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    # Línea de la media teórica
    ax.axhline(y=resultado["media_teorica"], color="blue",
               linestyle="--", label="Media teórica (0.5)")

    # Intervalo de confianza como franja
    ax.axhspan(resultado["limite_inferior"], resultado["limite_superior"],
               alpha=0.2, color="green", label="Intervalo de confianza 95%")

    # Líneas de los límites
    ax.axhline(y=resultado["limite_inferior"], color="green", linestyle=":", linewidth=1)
    ax.axhline(y=resultado["limite_superior"], color="green", linestyle=":", linewidth=1)

    # Punto de la media muestral
    color_punto = "blue" if resultado["aprueba"] else "red"
    ax.plot(0.5, resultado["media_muestral"], "o",
            color=color_punto, markersize=12,
            label=f"Media muestral ({resultado['media_muestral']})")

    # Etiquetas y formato
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title("Prueba de Medias — Secuencia única", fontsize=14, fontweight="bold")
    ax.set_ylabel("Valor", fontsize=12)
    ax.set_xticks([])
    ax.legend(loc="upper right")

    # Resultado en el gráfico
    texto = "APRUEBA ✓" if resultado["aprueba"] else "FALLA ✗"
    color_texto = "green" if resultado["aprueba"] else "red"
    ax.text(0.5, 0.1, texto, transform=ax.transAxes,
            fontsize=14, color=color_texto, ha="center", fontweight="bold")

    plt.tight_layout()
    plt.savefig("grafico_prueba_medias.png", dpi=150)
    plt.show()

In [ ]:
def grafico_distribucion_medias(lista_medias, nivel_confianza=0.95):
    """
    Gráfico para múltiples simulaciones: muestra la distribución
    de las medias calculadas de cada simulación (ej: 100 simulaciones de la rana).
    
    Parámetros:
        lista_medias: lista de medias, una por cada simulación
        nivel_confianza: nivel de confianza (default 95%)
    """
    N = len(lista_medias)
    media_global = sum(lista_medias) / N
    media_teorica = 0.5

    # Intervalo de confianza de la distribución de medias
    z = norm.ppf(1 - (1 - nivel_confianza) / 2)
    desviacion = (sum((m - media_global)**2 for m in lista_medias) / (N - 1)) ** 0.5
    margen = z * desviacion / math.sqrt(N)
    limite_inferior = media_teorica - margen
    limite_superior = media_teorica + margen

    fig, ax = plt.subplots(figsize=(10, 5))

    # Histograma de medias
    ax.hist(lista_medias, bins=20, color="steelblue", alpha=0.7,
            edgecolor="black", label="Distribución de medias")

    # Línea de la media teórica
    ax.axvline(x=media_teorica, color="blue", linestyle="--",
               linewidth=2, label="Media teórica (0.5)")

    # Línea de la media global calculada
    ax.axvline(x=media_global, color="red", linestyle="-",
               linewidth=2, label=f"Media global ({round(media_global, 4)})")

    # Intervalo de confianza
    ax.axvspan(limite_inferior, limite_superior, alpha=0.15,
               color="green", label=f"Intervalo de confianza 95%")

    # Formato
    ax.set_xlabel("Media de cada simulación", fontsize=12)
    ax.set_ylabel("Frecuencia", fontsize=12)
    ax.set_title("Prueba de Medias — Distribución de medias por simulación",
                 fontsize=14, fontweight="bold")
    ax.legend()

    # Información estadística
    aprueba = limite_inferior <= media_global <= limite_superior
    texto = "APRUEBA ✓" if aprueba else "FALLA ✗"
    color_texto = "green" if aprueba else "red"
    ax.text(0.98, 0.95,
            f"Simulaciones = {N}\nMedia global = {round(media_global, 6)}\n"
            f"LI = {round(limite_inferior, 6)}\nLS = {round(limite_superior, 6)}",
            transform=ax.transAxes, fontsize=10,
            verticalalignment="top", horizontalalignment="right",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
    ax.text(0.02, 0.95, texto, transform=ax.transAxes,
            fontsize=14, color=color_texto,
            verticalalignment="top", fontweight="bold")

    plt.tight_layout()
    plt.savefig("grafico_distribucion_medias.png", dpi=150)
    plt.show()

## 2. Prueba de Varianza
Verifica que la varianza de la secuencia pseudoaleatoria sea estadísticamente
cercana a 1/12 ≈ 0.0833, valor esperado de una distribución uniforme U(0,1).

In [ ]:
def prueba_varianza(secuencia, nivel_confianza=0.95):
    """
    Parámetros:
        secuencia: lista de números pseudoaleatorios entre 0 y 1
        nivel_confianza: nivel de confianza para la prueba (default 95%)
    Retorna:
        resultado: diccionario con todos los valores calculados
    """
    # Número de elementos en la secuencia
    N = len(secuencia)

    # Media muestral
    media = sum(secuencia) / N

    # Varianza muestral
    varianza_muestral = sum((x - media) ** 2 for x in secuencia) / (N - 1)

    # Varianza teórica para U(0,1)
    varianza_teorica = 1 / 12

    # Nivel de significancia y grados de libertad
    alpha = 1 - nivel_confianza
    gl = N - 1

    # Valores críticos Chi-cuadrado exactos 
    chi2_inferior = chi2.ppf(alpha / 2, gl)
    chi2_superior = chi2.ppf(1 - alpha / 2, gl)

    # Intervalo de confianza para la varianza
    limite_inferior = chi2_inferior / (12 * gl)
    limite_superior = chi2_superior / (12 * gl)

    # Verificar si la secuencia pasa la prueba
    aprueba = limite_inferior <= varianza_muestral <= limite_superior

    return {
        "N": N,
        "varianza_muestral": round(varianza_muestral, 6),
        "varianza_teorica": round(varianza_teorica, 6),
        "alpha": round(alpha, 4),
        "grados_libertad": gl,
        "chi2_inferior": round(chi2_inferior, 6),
        "chi2_superior": round(chi2_superior, 6),
        "limite_inferior": round(limite_inferior, 6),
        "limite_superior": round(limite_superior, 6),
        "aprueba": aprueba
    }

In [ ]:
def grafico_prueba_varianza(resultado):
    """
    Parámetros:
        resultado: diccionario retornado por prueba_varianza()
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    # Línea de la varianza teórica
    ax.axhline(y=resultado["varianza_teorica"], color="blue",
               linestyle="--", label=f"Varianza teórica (1/12 ≈ {resultado['varianza_teorica']})")

    # Intervalo de confianza como franja
    ax.axhspan(resultado["limite_inferior"], resultado["limite_superior"],
               alpha=0.2, color="green", label="Intervalo de confianza 95%")

    # Líneas de los límites
    ax.axhline(y=resultado["limite_inferior"], color="green", linestyle=":", linewidth=1)
    ax.axhline(y=resultado["limite_superior"], color="green", linestyle=":", linewidth=1)

    # Punto de la varianza muestral
    color_punto = "blue" if resultado["aprueba"] else "red"
    ax.plot(0.5, resultado["varianza_muestral"], "o",
            color=color_punto, markersize=12,
            label=f"Varianza muestral ({resultado['varianza_muestral']})")

    # Etiquetas y formato
    ax.set_xlim(0, 1)
    ax.set_ylim(0, resultado["limite_superior"] + 0.05)
    ax.set_title("Prueba de Varianza", fontsize=14, fontweight="bold")
    ax.set_ylabel("Valor", fontsize=12)
    ax.set_xticks([])
    ax.legend(loc="upper right")

    # Resultado en el gráfico
    texto = "APRUEBA ✓" if resultado["aprueba"] else "FALLA ✗"
    color_texto = "green" if resultado["aprueba"] else "red"
    ax.text(0.5, 0.1, texto, transform=ax.transAxes,
            fontsize=14, color=color_texto, ha="center", fontweight="bold")

    plt.tight_layout()
    plt.savefig("grafico_prueba_varianza.png", dpi=150)
    plt.show()

## 3. Prueba Chi-Cuadrado
Verifica que los números pseudoaleatorios se distribuyan de manera uniforme
en el intervalo [0,1], comparando las frecuencias observadas en cada
subintervalo contra las frecuencias esperadas teóricas.

In [ ]:
def prueba_chi_cuadrado(secuencia, k=10, nivel_confianza=0.95):
    """
    Parámetros:
        secuencia: lista de números pseudoaleatorios entre 0 y 1
        k: número de intervalos (default 10)
        nivel_confianza: nivel de confianza para la prueba (default 95%)
    Retorna:
        resultado: diccionario con todos los valores calculados
    """
    # Número de elementos en la secuencia
    N = len(secuencia)

    # Frecuencia esperada en cada intervalo
    frecuencia_esperada = N / k

    # Contar frecuencias observadas en cada intervalo
    frecuencias_observadas = [0] * k
    for numero in secuencia:
        indice = int(numero * k)
        if indice == k:  # caso borde: cuando número es exactamente 1.0
            indice = k - 1
        frecuencias_observadas[indice] += 1

    # Calcular estadístico chi-cuadrado
    chi2_calculado = sum(
        (obs - frecuencia_esperada) ** 2 / frecuencia_esperada
        for obs in frecuencias_observadas
    )

    # Valor crítico chi-cuadrado exacto para k-1 grados de libertad
    gl = k - 1
    chi2_critico = chi2.ppf(nivel_confianza, gl)

    # Verificar si la secuencia pasa la prueba
    aprueba = chi2_calculado < chi2_critico

    return {
        "N": N,
        "k": k,
        "frecuencias_observadas": frecuencias_observadas,
        "frecuencia_esperada": round(frecuencia_esperada, 6),
        "chi2_calculado": round(chi2_calculado, 6),
        "chi2_critico": round(chi2_critico, 6),
        "grados_libertad": gl,
        "aprueba": aprueba
    }

In [ ]:
def grafico_prueba_chi_cuadrado(resultado):
    """
    Parámetros:
        resultado: diccionario retornado por prueba_chi_cuadrado()
    """
    fig, ax = plt.subplots(figsize=(10, 5))

    k = resultado["k"]
    intervalos = [f"{i/k:.1f}-{(i+1)/k:.1f}" for i in range(k)]
    frecuencias_observadas = resultado["frecuencias_observadas"]
    frecuencia_esperada = resultado["frecuencia_esperada"]

    x = range(k)
    ancho = 0.35

    barras_obs = ax.bar([i - ancho/2 for i in x], frecuencias_observadas,
                        ancho, label="Frecuencia observada", color="steelblue")
    barras_esp = ax.bar([i + ancho/2 for i in x], [frecuencia_esperada] * k,
                        ancho, label="Frecuencia esperada", color="orange", alpha=0.7)

    for barra in barras_obs:
        ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.1,
                str(int(barra.get_height())), ha="center", va="bottom", fontsize=9)

    ax.set_xticks(list(x))
    ax.set_xticklabels(intervalos, rotation=45, ha="right")
    ax.set_xlabel("Intervalos", fontsize=12)
    ax.set_ylabel("Frecuencia", fontsize=12)
    ax.set_title("Prueba Chi-Cuadrado: Frecuencias Observadas vs Esperadas",
                 fontsize=14, fontweight="bold")
    ax.legend()

    ax.text(0.98, 0.95,
            f"χ² calculado = {resultado['chi2_calculado']}\n"
            f"χ² crítico = {resultado['chi2_critico']}\n"
            f"Grados de libertad = {resultado['grados_libertad']}",
            transform=ax.transAxes, fontsize=10,
            verticalalignment="top", horizontalalignment="right",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

    texto = "APRUEBA ✓" if resultado["aprueba"] else "FALLA ✗"
    color_texto = "green" if resultado["aprueba"] else "red"
    ax.text(0.02, 0.95, texto, transform=ax.transAxes,
            fontsize=14, color=color_texto, verticalalignment="top", fontweight="bold")

    plt.tight_layout()
    plt.savefig("grafico_prueba_chi_cuadrado.png", dpi=150)
    plt.show()

## 4. Prueba de Kolmogorov-Smirnov (KS)
Verifica que la distribución acumulada de la secuencia pseudoaleatoria
se ajuste a la distribución uniforme teórica U(0,1), comparando las
frecuencias acumuladas observadas contra las esperadas por intervalo
y calculando la máxima diferencia entre ambas.

In [ ]:
def prueba_ks(secuencia, k=10, nivel_confianza=0.95):
    """
    Parámetros:
        secuencia: lista de números pseudoaleatorios entre 0 y 1
        k: número de intervalos (default 10)
        nivel_confianza: nivel de confianza para la prueba (default 95%)
    Retorna:
        resultado: diccionario con todos los valores calculados
    """
    # Número de elementos en la secuencia
    N = len(secuencia)

    # Frecuencia esperada por intervalo
    frec_esperada = N / k

    # Contar frecuencias observadas por intervalo
    frecuencias_observadas = [0] * k
    for numero in secuencia:
        indice = int(numero * k)
        if indice == k:  # caso borde: cuando número es exactamente 1.0
            indice = k - 1
        frecuencias_observadas[indice] += 1

    # Calcular frecuencias acumuladas y diferencias
    tabla = []
    frec_obs_acumulada = 0
    frec_esp_acumulada = 0

    for i in range(k):
        limite_inicial = round(i / k, 2)
        limite_final = round((i + 1) / k, 2)

        frec_obs_acumulada += frecuencias_observadas[i]
        frec_esp_acumulada += frec_esperada

        p_obs_acumulada = frec_obs_acumulada / N
        p_esp_acumulada = frec_esp_acumulada / N
        diferencia = abs(p_obs_acumulada - p_esp_acumulada)

        tabla.append({
            "intervalo": f"{limite_inicial}-{limite_final}",
            "frec_observada": frecuencias_observadas[i],
            "frec_obs_acumulada": frec_obs_acumulada,
            "p_obs_acumulada": round(p_obs_acumulada, 6),
            "frec_esp_acumulada": frec_esp_acumulada,
            "p_esp_acumulada": round(p_esp_acumulada, 6),
            "diferencia": round(diferencia, 6)
        })

    # DMAX: la diferencia más grande
    dmax = max(fila["diferencia"] for fila in tabla)

    # DMAXP: valor crítico (equivalente al de la tabla del docente)
    dmaxp = 1.36 / math.sqrt(N)

    # Verificar si la secuencia pasa la prueba
    aprueba = dmax < dmaxp

    return {
        "N": N,
        "k": k,
        "tabla": tabla,
        "dmax": round(dmax, 6),
        "dmaxp": round(dmaxp, 6),
        "aprueba": aprueba
    }

In [ ]:
def grafico_prueba_ks(resultado):
    """
    Parámetros:
        resultado: diccionario retornado por prueba_ks()
    """
    fig, ax = plt.subplots(figsize=(10, 5))

    intervalos = [fila["intervalo"] for fila in resultado["tabla"]]
    p_obs = [fila["p_obs_acumulada"] for fila in resultado["tabla"]]
    p_esp = [fila["p_esp_acumulada"] for fila in resultado["tabla"]]
    diferencias = [fila["diferencia"] for fila in resultado["tabla"]]

    ax.plot(intervalos, p_obs, marker="o", color="steelblue",
            linewidth=2, label="P. Acumulada Observada")
    ax.plot(intervalos, p_esp, marker="s", color="orange",
            linewidth=2, linestyle="--", label="P. Acumulada Esperada")

    indice_dmax = diferencias.index(max(diferencias))
    ax.annotate(f"DMAX = {resultado['dmax']}",
                xy=(intervalos[indice_dmax], p_obs[indice_dmax]),
                xytext=(indice_dmax + 0.5, p_obs[indice_dmax] - 0.1),
                arrowprops=dict(arrowstyle="->", color="red"),
                fontsize=10, color="red")

    ax.set_xticks(range(len(intervalos)))
    ax.set_xticklabels(intervalos, rotation=45, ha="right")
    ax.set_xlabel("Intervalos", fontsize=12)
    ax.set_ylabel("Proporción Acumulada", fontsize=12)
    ax.set_title("Prueba Kolmogorov-Smirnov: FEC Observada vs Esperada",
                 fontsize=14, fontweight="bold")
    ax.legend()

    ax.text(0.02, 0.95,
            f"DMAX = {resultado['dmax']}\nDMAXP = {resultado['dmaxp']}\nn = {resultado['N']}",
            transform=ax.transAxes, fontsize=10, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

    texto = "APRUEBA ✓" if resultado["aprueba"] else "FALLA ✗"
    color_texto = "green" if resultado["aprueba"] else "red"
    ax.text(0.98, 0.05, texto, transform=ax.transAxes,
            fontsize=14, color=color_texto, ha="right", fontweight="bold")

    plt.tight_layout()
    plt.savefig("grafico_prueba_ks.png", dpi=150)
    plt.show()

## 5. Prueba de Póker
Verifica la independencia de los números pseudoaleatorios analizando
los patrones formados por sus 5 dígitos decimales significativos,
comparando las frecuencias observadas de cada categoría contra
las frecuencias esperadas teóricas mediante un estadístico Chi-cuadrado.

In [ ]:
def prueba_poker(secuencia, nivel_confianza=0.95):
    """
    Parámetros:
        secuencia: lista de números pseudoaleatorios entre 0 y 1
        nivel_confianza: nivel de confianza para la prueba (default 95%)
    Retorna:
        resultado: diccionario con todos los valores calculados
    """
    # Número de elementos en la secuencia
    N = len(secuencia)

    # Probabilidades teóricas de cada categoría (según docente)
    categorias = {
        "D - Todos diferentes": 0.3024,
        "O - Un par":           0.5040,
        "T - Dos Pares":        0.1080,
        "K - Tercia":           0.0720,
        "F - Full House":       0.0090,
        "P - Poker":            0.0045,
        "Q - Quintilla":        0.0001
    }

    # Frecuencias observadas inicializadas en 0
    frecuencias_observadas = {cat: 0 for cat in categorias}

    # Clasificar cada número según sus 5 dígitos significativos
    for numero in secuencia:
        digitos = str(int(round(numero * 100000))).zfill(5)[:5]
        conteo = {}
        for d in digitos:
            conteo[d] = conteo.get(d, 0) + 1
        frecuencias = sorted(conteo.values(), reverse=True)

        if frecuencias[0] == 5:
            frecuencias_observadas["Q - Quintilla"] += 1
        elif frecuencias[0] == 4:
            frecuencias_observadas["P - Poker"] += 1
        elif frecuencias[0] == 3 and frecuencias[1] == 2:
            frecuencias_observadas["F - Full House"] += 1
        elif frecuencias[0] == 3:
            frecuencias_observadas["K - Tercia"] += 1
        elif frecuencias[0] == 2 and len(frecuencias) > 1 and frecuencias[1] == 2:
            frecuencias_observadas["T - Dos Pares"] += 1
        elif frecuencias[0] == 2:
            frecuencias_observadas["O - Un par"] += 1
        else:
            frecuencias_observadas["D - Todos diferentes"] += 1

    # Calcular frecuencias esperadas y estadístico chi-cuadrado
    tabla = []
    chi2_calculado = 0

    for cat, prob in categorias.items():
        obs = frecuencias_observadas[cat]
        esp = N * prob
        contribucion = (obs - esp) ** 2 / esp
        chi2_calculado += contribucion
        tabla.append({
            "categoria": cat,
            "frecuencia_observada": obs,
            "probabilidad": prob,
            "frecuencia_esperada": round(esp, 4),
            "contribucion_chi2": round(contribucion, 6)
        })

    # Valor crítico chi-cuadrado para 6 grados de libertad
    chi2_critico = chi2.ppf(nivel_confianza, 6)

    # Verificar si la secuencia pasa la prueba
    aprueba = chi2_calculado < chi2_critico

    return {
        "N": N,
        "tabla": tabla,
        "chi2_calculado": round(chi2_calculado, 6),
        "chi2_critico": round(chi2_critico, 6),
        "grados_libertad": 6,
        "aprueba": aprueba
    }

In [ ]:
def grafico_prueba_poker(resultado):
    """
    Parámetros:
        resultado: diccionario retornado por prueba_poker()
    """
    fig, ax = plt.subplots(figsize=(10, 5))

    categorias = [fila["categoria"] for fila in resultado["tabla"]]
    frec_obs = [fila["frecuencia_observada"] for fila in resultado["tabla"]]
    frec_esp = [fila["frecuencia_esperada"] for fila in resultado["tabla"]]

    x = range(len(categorias))
    ancho = 0.35

    barras_obs = ax.bar([i - ancho/2 for i in x], frec_obs,
                        ancho, label="Frecuencia observada", color="steelblue")
    barras_esp = ax.bar([i + ancho/2 for i in x], frec_esp,
                        ancho, label="Frecuencia esperada", color="orange", alpha=0.7)

    for barra in barras_obs:
        ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.5,
                str(int(barra.get_height())), ha="center", va="bottom", fontsize=9)

    ax.set_xticks(list(x))
    ax.set_xticklabels(categorias, rotation=45, ha="right")
    ax.set_xlabel("Categorías", fontsize=12)
    ax.set_ylabel("Frecuencia", fontsize=12)
    ax.set_title("Prueba de Póker: Frecuencias Observadas vs Esperadas",
                 fontsize=14, fontweight="bold")
    ax.legend()

    ax.text(0.98, 0.95,
            f"χ² calculado = {resultado['chi2_calculado']}\n"
            f"χ² crítico = {resultado['chi2_critico']}\n"
            f"Grados de libertad = {resultado['grados_libertad']}",
            transform=ax.transAxes, fontsize=10,
            verticalalignment="top", horizontalalignment="right",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

    texto = "APRUEBA ✓" if resultado["aprueba"] else "FALLA ✗"
    color_texto = "green" if resultado["aprueba"] else "red"
    ax.text(0.02, 0.95, texto, transform=ax.transAxes,
            fontsize=14, color=color_texto, verticalalignment="top", fontweight="bold")

    plt.tight_layout()
    plt.savefig("grafico_prueba_poker.png", dpi=150)
    plt.show()

## 6. Prueba de Rachas
Verifica la aleatoriedad de la secuencia pseudoaleatoria detectando
patrones de agrupamiento por encima o por debajo de la mediana.
Clasifica cada número como (+) si es mayor o igual a la mediana
y (-) si es menor, contando las rachas resultantes y comparando
contra los valores esperados teóricos mediante un estadístico Z.

In [ ]:
def prueba_rachas(secuencia, nivel_confianza=0.95):
    """
    Parámetros:
        secuencia: lista de números pseudoaleatorios entre 0 y 1
        nivel_confianza: nivel de confianza para la prueba (default 95%)
    Retorna:
        resultado: diccionario con todos los valores calculados
    """
    # Número de elementos en la secuencia
    N = len(secuencia)

    # Calcular la mediana de la secuencia
    secuencia_ordenada = sorted(secuencia)
    if N % 2 == 0:
        mediana = (secuencia_ordenada[N//2 - 1] + secuencia_ordenada[N//2]) / 2
    else:
        mediana = secuencia_ordenada[N//2]

    # Mediana teórica para U(0,1)
    mediana_teorica = 0.5

    # Clasificar cada número como + o -
    signos = []
    for numero in secuencia:
        if numero >= mediana:
            signos.append("+")
        else:
            signos.append("-")

    # Contar total de + y total de -
    n1 = signos.count("+")
    n2 = signos.count("-")

    # Contar rachas totales
    rachas = 1
    for i in range(1, len(signos)):
        if signos[i] != signos[i-1]:
            rachas += 1

    # Rachas esperadas y varianza esperada
    rachas_esperadas = ((2 * n1 * n2) / N) + 1
    varianza_esperada = (2 * n1 * n2 * (2 * n1 * n2 - N)) / (N**2 * (N - 1))

    # Estadístico Z
    z_calculado = (rachas - rachas_esperadas) / math.sqrt(varianza_esperada)

    # Valor crítico Z calculado dinámicamente según nivel de confianza
    z_critico = norm.ppf(1 - (1 - nivel_confianza) / 2)

    # Rango de aceptación
    rango_minimo = -z_critico
    rango_maximo = z_critico

    # Verificar si la secuencia pasa la prueba
    aprueba = rango_minimo <= z_calculado <= rango_maximo

    return {
        "N": N,
        "mediana": round(mediana, 6),
        "mediana_teorica": mediana_teorica,
        "n1_positivos": n1,
        "n2_negativos": n2,
        "rachas_observadas": rachas,
        "rachas_esperadas": round(rachas_esperadas, 6),
        "varianza_esperada": round(varianza_esperada, 6),
        "z_calculado": round(z_calculado, 6),
        "z_critico": round(z_critico, 6),
        "rango_minimo": round(rango_minimo, 6),
        "rango_maximo": round(rango_maximo, 6),
        "aprueba": aprueba
    }

In [ ]:
def grafico_prueba_rachas(resultado):
    """
    Parámetros:
        resultado: diccionario retornado por prueba_rachas()
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    categorias = ["Rachas\nObservadas", "Rachas\nEsperadas"]
    valores = [resultado["rachas_observadas"], resultado["rachas_esperadas"]]
    colores = ["steelblue", "orange"]

    barras = ax.bar(categorias, valores, color=colores,
                    width=0.4, edgecolor="black", linewidth=0.8)

    margen = resultado["z_critico"] * math.sqrt(resultado["varianza_esperada"])
    ax.errorbar(1, resultado["rachas_esperadas"],
                yerr=margen, fmt="none", color="red", capsize=8, linewidth=2,
                label=f"Intervalo de confianza 95%\n(±{round(margen, 2)})")

    for barra, valor in zip(barras, valores):
        ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.3,
                str(round(valor, 2)), ha="center", va="bottom",
                fontsize=11, fontweight="bold")

    ax.set_ylabel("Número de Rachas", fontsize=12)
    ax.set_title("Prueba de Rachas: Observadas vs Esperadas",
                 fontsize=14, fontweight="bold")
    ax.legend(loc="upper right")
    ax.set_ylim(0, max(valores) + 15)

    ax.text(0.02, 0.95,
            f"Z calculado = {resultado['z_calculado']}\n"
            f"Z crítico = ±{resultado['z_critico']}\n"
            f"n1(+) = {resultado['n1_positivos']}\n"
            f"n2(-) = {resultado['n2_negativos']}\n"
            f"Mediana = {resultado['mediana']}",
            transform=ax.transAxes, fontsize=10, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

    texto = "APRUEBA ✓" if resultado["aprueba"] else "FALLA ✗"
    color_texto = "green" if resultado["aprueba"] else "red"
    ax.text(0.98, 0.05, texto, transform=ax.transAxes,
            fontsize=14, color=color_texto, ha="right", fontweight="bold")

    plt.tight_layout()
    plt.savefig("grafico_prueba_rachas.png", dpi=150)
    plt.show()

## 7. Uso del módulo
Esta sección muestra cómo conectar el módulo con los generadores
y cómo usar `grafico_distribucion_medias` con múltiples simulaciones.

In [ ]:
# ============================================================
# EJEMPLO DE USO CON GENERADORES DE PERSONA A
# Reemplazar 'secuencia' por la salida del generador correspondiente
# ============================================================

# Ejemplo: secuencia = generador_congruencial_lineal(semilla=42, n=1000)
# resultado_medias   = prueba_medias(secuencia)
# resultado_varianza = prueba_varianza(secuencia)
# resultado_chi      = prueba_chi_cuadrado(secuencia)
# resultado_ks       = prueba_ks(secuencia)
# resultado_poker    = prueba_poker(secuencia)
# resultado_rachas   = prueba_rachas(secuencia)

# ============================================================
# EJEMPLO DE USO CON MÚLTIPLES SIMULACIONES (Punto 2 - La rana)
# lista_medias contiene la media de cada simulación independiente
# ============================================================

# Ejemplo: lista_medias = [mean(sim) for sim in simulaciones_rana]
# grafico_distribucion_medias(lista_medias)

print("Módulo de validación cargado correctamente.")
print("Funciones disponibles:")
print("  - prueba_medias(secuencia)")
print("  - prueba_varianza(secuencia)")
print("  - prueba_chi_cuadrado(secuencia)")
print("  - prueba_ks(secuencia)")
print("  - prueba_poker(secuencia)")
print("  - prueba_rachas(secuencia)")
print("  - grafico_distribucion_medias(lista_medias)  ← para múltiples simulaciones")